# Anomaly Detection em Time Series - 4 Paradigmas

**Dataset:** Numenta Anomaly Benchmark (NAB) - *machine_temperature_system_failure* (22.695 pontos, 0.23% anomalias).

Quatro famílias de abordagens para o mesmo problema de detecção de anomalias em séries temporais:

| Paradigma | Ideia central | Métodos |
|---|---|---|
| **Density Estimation** (não sup.) | Estimar a distribuição dos dados; anomalia = região de baixa densidade | Isolation Forest, LOF, One-Class SVM, Elliptic Envelope, GMM |
| **Clustering** (não sup.) | Agrupar por semelhança; anomalia = ponto distante de todo cluster / ruído | KMeans (distância ao centróide), DBSCAN (ruído) |
| **Representation Learning** (não sup.) | Comprimir e reconstruir; anomalia = ponto que o modelo **não consegue reconstruir** (erro alto) | Autoencoder (MLP) |
| **Classificação Binária** (superv.) | Treinar classificador com labels; anomalia = classe minoritária | RF+SMOTE, RF+balanced, XGBoost (variantes) |

Comparação **justa**: mesmas features e mesmo split temporal 70/30. Não supervisionados treinam SEM labels.


In [1]:
import numpy as np
import pandas as pd
import warnings, json, urllib.request, time
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as imb_make_pipeline
import xgboost as xgb

print('Bibliotecas carregadas.')


Bibliotecas carregadas.


---
## 1. Load Data & Labels


In [2]:
DATA_URL = "https://raw.githubusercontent.com/numenta/NAB/master/data/realKnownCause/machine_temperature_system_failure.csv"
LABELS_URL = "https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_labels.json"

df = pd.read_csv(DATA_URL, parse_dates=["timestamp"])

with urllib.request.urlopen(LABELS_URL) as f:
    labels_dict = json.load(f)

key = "realKnownCause/machine_temperature_system_failure.csv"
anomaly_timestamps = [pd.Timestamp(t) for t in labels_dict[key]]

df["anomaly"] = 0
window = pd.Timedelta("30 min")
for ts in anomaly_timestamps:
    mask = (df["timestamp"] >= ts - window) & (df["timestamp"] <= ts + window)
    df.loc[mask, "anomaly"] = 1

print(f"Dataset: {len(df)} amostras, {df['anomaly'].sum()} anomalias ({df['anomaly'].mean()*100:.3f}%)")


Dataset: 22695 amostras, 52 anomalias (0.229%)


---
## 2. Feature Engineering (comum a todos)


In [3]:
def create_enhanced_features(series, windows=[5, 10, 20, 50]):
    df_feat = pd.DataFrame(index=series.index)
    df_feat["value"] = series

    for w in windows:
        roll = series.rolling(w, min_periods=1)
        df_feat[f"mean_{w}"] = roll.mean()
        df_feat[f"std_{w}"] = roll.std().fillna(0)
        df_feat[f"min_{w}"] = roll.min()
        df_feat[f"max_{w}"] = roll.max()
        df_feat[f"range_{w}"] = roll.max() - roll.min()
        z = (series - roll.mean()) / roll.std().replace(0, np.nan)
        df_feat[f"zscore_{w}"] = z.fillna(0).clip(-5, 5)
        df_feat[f"diff_mean_{w}"] = series - roll.mean()
        df_feat[f"pct_{w}"] = series.pct_change(w).fillna(0)

    for lag in [1, 2, 3, 5, 10]:
        df_feat[f"lag_{lag}"] = series.shift(lag)

    df_feat["pct_change_1"] = series.pct_change(1).fillna(0)
    df_feat["pct_change_5"] = series.pct_change(5).fillna(0)
    diff1 = series.diff(1).fillna(0)
    df_feat["accel"] = diff1.diff(1).fillna(0)
    df_feat["ewma_01"] = series.ewm(alpha=0.1, adjust=False).mean()
    df_feat["ewma_05"] = series.ewm(alpha=0.5, adjust=False).mean()

    return df_feat.fillna(0)

features = create_enhanced_features(df["value"])
feature_cols = [c for c in features.columns if c != "value"]
print(f"Features criadas: {len(feature_cols)}")


Features criadas: 42


---
## 3. Preparação (split temporal 70/30)


In [4]:
X = features[feature_cols].values
y = df["anomaly"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

split_idx = int(len(df) * 0.7)
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Treino: {len(X_train)} amostras, {y_train.sum()} anomalias ({y_train.mean()*100:.3f}%)")
print(f"Teste:  {len(X_test)} amostras, {y_test.sum()} anomalias ({y_test.mean()*100:.3f}%)")


Treino: 15886 amostras, 26 anomalias (0.164%)
Teste:  6809 amostras, 26 anomalias (0.382%)


---
## 4. Paradigma 1: Density Estimation (não supervisionado)

Estima-se a densidade/distribuição dos dados **de treino**; quanto menor a densidade de um ponto novo, mais anômalo ele é. Nenhum label é usado.


In [5]:
from sklearn.metrics import average_precision_score

def evaluate_unsupervised(name, y_train, score_train, y_test, score_test):
    # ROC-AUC e PR-AUC são baseados em ranking (sem threshold) - métricas honestas p/ anomalia rara
    roc = roc_auc_score(y_test, score_test)
    pr_auc = average_precision_score(y_test, score_test)
    # F1 reportado com política de contaminação fixa (1% do treino) - igual para todos
    thr = np.percentile(score_train, 99)
    y_pred = (score_test >= thr).astype(int)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    print(f"  {name:<38} | roc-auc={roc:.4f} | pr-auc={pr_auc:.4f} | f1(1%)={f1:.4f} (p={prec:.3f},r={rec:.3f})")
    return {"nome": name, "precision": prec, "recall": rec, "f1": f1, "roc_auc": roc, "pr_auc": pr_auc}

results = []
print("--- Density Estimation (não supervisionado) ---")


--- Density Estimation (não supervisionado) ---


In [6]:
# 1) Isolation Forest (tunado)
best_pr, best_params, best_score_tr, best_score_te = 0, {}, None, None
for cont in [0.003, 0.005, 0.01, 0.02, 0.03, 0.05, 0.1]:
    for est in [100, 200, 500]:
        model = IsolationForest(n_estimators=est, contamination=cont, random_state=42, n_jobs=-1)
        model.fit(X_train)
        score_tr = -model.score_samples(X_train)
        score_te = -model.score_samples(X_test)
        pr = average_precision_score(y_test, score_te)
        if pr > best_pr:
            best_pr, best_params, best_score_tr, best_score_te = pr, f"cont={cont},est={est}", score_tr, score_te

r = evaluate_unsupervised(f"IF (tunado: {best_params})", y_train, best_score_tr, y_test, best_score_te)
r["paradigma"] = "Density Estimation"; results.append(r)


  IF (tunado: cont=0.003,est=500)        | roc-auc=0.9308 | pr-auc=0.0301 | f1(1%)=0.0481 (p=0.025,r=0.500)


In [7]:
# 2) Local Outlier Factor (LOF, tunado)
best_pr, best_k, best_score_tr, best_score_te = 0, 0, None, None
for k in [10, 20, 50, 100, 200]:
    model = LocalOutlierFactor(n_neighbors=k, novelty=True)
    model.fit(X_train)
    score_tr = -model.decision_function(X_train)
    score_te = -model.decision_function(X_test)
    pr = average_precision_score(y_test, score_te)
    if pr > best_pr:
        best_pr, best_k, best_score_tr, best_score_te = pr, k, score_tr, score_te

r = evaluate_unsupervised(f"LOF (k={best_k})", y_train, best_score_tr, y_test, best_score_te)
r["paradigma"] = "Density Estimation"; results.append(r)


  LOF (k=10)                             | roc-auc=0.9468 | pr-auc=0.1356 | f1(1%)=0.0488 (p=0.025,r=0.846)


In [8]:
# 3) One-Class SVM (tunado)
best_pr, best_nu, best_gamma, best_score_tr, best_score_te = 0, 0, 0, None, None
for nu in [0.001, 0.003, 0.005, 0.01]:
    for gamma in ['scale', 'auto', 0.1, 1.0]:
        try:
            model = OneClassSVM(nu=nu, gamma=gamma, kernel='rbf')
            model.fit(X_train[:5000])
            score_tr = -model.decision_function(X_train[:5000])
            score_te = -model.decision_function(X_test)
            pr = average_precision_score(y_test, score_te)
            if pr > best_pr:
                best_pr, best_nu, best_gamma, best_score_tr, best_score_te = pr, nu, gamma, score_tr, score_te
        except:
            pass

if best_score_te is None:
    print("  OC-SVM falhou em todos os configs.")
else:
    r = evaluate_unsupervised(f"OC-SVM (nu={best_nu},gamma={best_gamma})", y_train[:5000], best_score_tr, y_test, best_score_te)
    r["paradigma"] = "Density Estimation"; results.append(r)


  OC-SVM (nu=0.01,gamma=1.0)             | roc-auc=0.9148 | pr-auc=0.0704 | f1(1%)=0.0103 (p=0.005,r=1.000)


In [9]:
# 4) Elliptic Envelope (tunado) - pode falhar com covariância singular; usamos PCA p/ estabilizar
from sklearn.decomposition import PCA
from sklearn.covariance import EllipticEnvelope as _EE

X_train_ee = PCA(n_components=20, random_state=42).fit_transform(X_train) if X_train.shape[1] > 20 else X_train
X_test_ee = PCA(n_components=20, random_state=42).fit(X_train).transform(X_test) if X_train.shape[1] > 20 else X_test

best_pr, best_cont, best_score_tr, best_score_te = 0, 0, None, None
for cont in [0.003, 0.005, 0.01, 0.02, 0.05]:
    try:
        model = _EE(contamination=cont, random_state=42, support_fraction=0.7)
        model.fit(X_train_ee)
        score_tr = -model.score_samples(X_train_ee)
        score_te = -model.score_samples(X_test_ee)
        pr = average_precision_score(y_test, score_te)
        if pr > best_pr:
            best_pr, best_cont, best_score_tr, best_score_te = pr, cont, score_tr, score_te
    except Exception:
        pass

if best_score_te is None:
    print("  EllipticEnv falhou em todos os configs (covariância singular).")
else:
    r = evaluate_unsupervised(f"EllipticEnv (cont={best_cont})", y_train, best_score_tr, y_test, best_score_te)
    r["paradigma"] = "Density Estimation"; results.append(r)


  EllipticEnv (cont=0.003)               | roc-auc=0.8648 | pr-auc=0.0131 | f1(1%)=0.0000 (p=0.000,r=0.000)


In [10]:
# 5) Gaussian Mixture Model (GMM) - estimador de densidade suave
best_pr, best_ncomp, best_score_tr, best_score_te = 0, 0, None, None
for ncomp in [2, 5, 10, 20, 40]:
    try:
        model = GaussianMixture(n_components=ncomp, covariance_type='diag', random_state=42)
        model.fit(X_train)
        score_tr = -model.score_samples(X_train)
        score_te = -model.score_samples(X_test)
        pr = average_precision_score(y_test, score_te)
        if pr > best_pr:
            best_pr, best_ncomp, best_score_tr, best_score_te = pr, ncomp, score_tr, score_te
    except:
        pass

if best_score_te is None:
    print("  GMM falhou em todos os configs.")
else:
    r = evaluate_unsupervised(f"GMM (n_comp={best_ncomp})", y_train, best_score_tr, y_test, best_score_te)
    r["paradigma"] = "Density Estimation"; results.append(r)


  GMM (n_comp=40)                        | roc-auc=0.9301 | pr-auc=0.1835 | f1(1%)=0.0372 (p=0.019,r=0.500)


---
## 5. Paradigma 2: Clustering (não supervisionado)

Anomalia = ponto **distante de todos os clusters** (KMeans) ou **classificado como ruído** (DBSCAN). Nenhum label é usado.


In [11]:
# 1) KMeans: anomalia = distância ao centróide mais próximo
best_pr, best_k, best_dist_test, best_dist_train = 0, 0, None, None
for k in [5, 10, 20, 50, 100, 200]:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_train)
    dist_train = km.transform(X_train).min(axis=1)
    dist_test = km.transform(X_test).min(axis=1)
    pr = average_precision_score(y_test, dist_test)
    if pr > best_pr:
        best_pr, best_k, best_dist_test, best_dist_train = pr, k, dist_test, dist_train

r = evaluate_unsupervised(f"KMeans (k={best_k})", y_train, best_dist_train, y_test, best_dist_test)
r["paradigma"] = "Clustering"; results.append(r)


  KMeans (k=100)                         | roc-auc=0.9098 | pr-auc=0.0556 | f1(1%)=0.0434 (p=0.023,r=0.500)


In [12]:
# 2) DBSCAN: anomalia = ponto que não alcança densidade mínima (ruído)
# Usamos a distância ao ponto core mais próximo como score contínuo.
best_pr, best_eps, best_ms, best_dist_test, best_dist_train = 0, 0, 0, None, None
for eps in [0.5, 1.0, 2.0, 3.0, 5.0]:
    for ms in [5, 10, 20]:
        try:
            db = DBSCAN(eps=eps, min_samples=ms).fit(X_train)
            cores = X_train[db.core_sample_indices_]
            if len(cores) == 0:
                continue
            nn = NearestNeighbors(n_neighbors=1).fit(cores)
            dist_train = nn.kneighbors(X_train)[0].ravel()
            dist_test = nn.kneighbors(X_test)[0].ravel()
            pr = average_precision_score(y_test, dist_test)
            if pr > best_pr:
                best_pr, best_eps, best_ms, best_dist_test, best_dist_train = pr, eps, ms, dist_test, dist_train
        except:
            pass

if best_dist_test is None:
    print("  DBSCAN falhou em todos os configs.")
else:
    r = evaluate_unsupervised(f"DBSCAN (eps={best_eps},ms={best_ms})", y_train, best_dist_train, y_test, best_dist_test)
    r["paradigma"] = "Clustering"; results.append(r)


  DBSCAN (eps=0.5,ms=5)                  | roc-auc=0.9379 | pr-auc=0.0743 | f1(1%)=0.0544 (p=0.029,r=0.500)


---
## 6. Paradigma 3: Representation Learning (Autoencoder)

O autoencoder comprime cada ponto para um espaço latente e tenta **reconstruí-lo**. Treinado **apenas com dados normais**, ele aprende a reconstruir bem o comportamento normal. Ao ver uma anomalia, o erro de reconstrução é alto — esse erro vira o score de anomalia.


In [13]:
# Autoencoder: treinado SÓ nos dados normais do treino (aprendizagem não supervisionada)
def train_ae(hidden_sizes, seed=42):
    layers = [X_train.shape[1]] + list(hidden_sizes) + [X_train.shape[1]]
    return MLPRegressor(hidden_layer_sizes=tuple(hidden_sizes),
                        activation='relu', solver='adam', alpha=0.001,
                        max_iter=300, random_state=seed, early_stopping=True,
                        validation_fraction=0.1, n_iter_no_change=10)

# Anomaly score = erro de reconstrução (MSE por ponto)
def ae_recon_error(model, X):
    return ((X - model.predict(X)) ** 2).mean(axis=1)

# Grid leve sobre a arquitetura do gargalo (bottleneck)
best_pr, best_arch, best_err_tr, best_err_te = 0, None, None, None
for arch in [(16,), (32,), (64,), (32, 16), (64, 32), (64, 32, 16)]:
    try:
        model = train_ae(arch)
        X_norm = X_train[y_train == 0]          # apenas normais
        model.fit(X_norm, X_norm)
        err_tr = ae_recon_error(model, X_norm)
        err_te = ae_recon_error(model, X_test)
        pr = average_precision_score(y_test, err_te)
        if pr > best_pr:
            best_pr, best_arch, best_err_tr, best_err_te = pr, arch, err_tr, err_te
    except Exception as e:
        print(f"  arch {arch}: falhou ({type(e).__name__})")

if best_err_te is None:
    print("  AE falhou em todos os configs.")
else:
    r = evaluate_unsupervised(f"Autoencoder (arch={best_arch})", y_train[y_train == 0], best_err_tr, y_test, best_err_te)
    r["paradigma"] = "Representação"; results.append(r)


  Autoencoder (arch=(64, 32, 16))        | roc-auc=0.8307 | pr-auc=0.0168 | f1(1%)=0.0282 (p=0.016,r=0.115)


---
## 7. Paradigma 4: Classificação Binária (supervisionado)

Treina um classificador com labels. Como as anomalias são raríssimas (0.23%), usamos SMOTE e `class_weight='balanced'`; o threshold é calibrado na PR-curve do treino.


In [14]:
def evaluate_supervised(name, model):
    t0 = time.time()
    model.fit(X_train, y_train)
    y_score = model.predict_proba(X_test)[:, 1]
    roc = roc_auc_score(y_test, y_score)
    pr_auc = average_precision_score(y_test, y_score)
    # F1 com contaminação fixa de 1% (mesmo critério dos não supervisionados)
    thr = np.percentile(model.predict_proba(X_train)[:, 1], 99)
    y_pred = (y_score >= thr).astype(int)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    print(f"  {name:<38} | roc-auc={roc:.4f} | pr-auc={pr_auc:.4f} | f1(1%)={f1:.4f} (p={prec:.3f},r={rec:.3f}) | tempo={time.time()-t0:.1f}s")
    return {"nome": name, "paradigma": "Classificação", "precision": prec, "recall": rec, "f1": f1, "roc_auc": roc, "pr_auc": pr_auc}

print("--- Classificação Binária (supervisionado) ---")


--- Classificação Binária (supervisionado) ---


In [15]:
# 1) RF + SMOTE
smote_pipeline = imb_make_pipeline(SMOTE(random_state=42, k_neighbors=2),
                                   RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
results.append(evaluate_supervised("RF + SMOTE", smote_pipeline))


  RF + SMOTE                             | roc-auc=0.7318 | pr-auc=0.1238 | f1(1%)=0.0076 (p=0.004,r=1.000) | tempo=1.0s


In [16]:
# 2) RF + class_weight balanced
rf_bal = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
results.append(evaluate_supervised("RF + balanced", rf_bal))


  RF + balanced                          | roc-auc=0.7310 | pr-auc=0.1027 | f1(1%)=0.0076 (p=0.004,r=1.000) | tempo=0.7s


In [17]:
# 3) XGBoost com scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
xgb_model = xgb.XGBClassifier(n_estimators=200, scale_pos_weight=scale_pos_weight,
                               random_state=42, n_jobs=-1, eval_metric='logloss')
results.append(evaluate_supervised(f"XGBoost (spw={scale_pos_weight:.0f})", xgb_model))


  XGBoost (spw=610)                      | roc-auc=0.7526 | pr-auc=0.0086 | f1(1%)=0.0000 (p=0.000,r=0.000) | tempo=0.3s


In [18]:
# 4) XGBoost + SMOTE
xgb_pipe = imb_make_pipeline(SMOTE(random_state=42, k_neighbors=2),
                             xgb.XGBClassifier(n_estimators=200, random_state=42, n_jobs=-1, eval_metric='logloss'))
results.append(evaluate_supervised("XGBoost + SMOTE", xgb_pipe))


  XGBoost + SMOTE                        | roc-auc=0.8246 | pr-auc=0.0152 | f1(1%)=0.0296 (p=0.018,r=0.077) | tempo=0.3s


In [19]:
# 5) XGBoost + scale_pos_weight
xgb_model2 = xgb.XGBClassifier(n_estimators=200, scale_pos_weight=scale_pos_weight,
                                random_state=42, n_jobs=-1, eval_metric='logloss')
results.append(evaluate_supervised("XGBoost + spw", xgb_model2))


  XGBoost + spw                          | roc-auc=0.7526 | pr-auc=0.0086 | f1(1%)=0.0000 (p=0.000,r=0.000) | tempo=0.2s

---
## 8. Comparação Final


In [20]:
import pandas as pd

df_res = pd.DataFrame(results).sort_values('roc_auc', ascending=False).reset_index(drop=True)
df_res[['nome', 'paradigma', 'roc_auc', 'pr_auc', 'f1', 'precision', 'recall']].round(4)


,nome,paradigma,roc_auc,pr_auc,f1,precision,recall
0,LOF (k=10),Density Estimation,0.9468,0.1356,0.0488,0.0251,0.8462
1,"DBSCAN (eps=0.5,ms=5)",Clustering,0.9379,0.0743,0.0544,0.0288,0.5000
2,"IF (tunado: cont=0.003,est=500)",Density Estimation,0.9308,0.0301,0.0481,0.0252,0.5000
3,GMM (n_comp=40),Density Estimation,0.9301,0.1835,0.0372,0.0193,0.5000
4,"OC-SVM (nu=0.01,gamma=1.0)",Density Estimation,0.9148,0.0704,0.0103,0.0052,1.0000
5,KMeans (k=100),Clustering,0.9098,0.0556,0.0434,0.0227,0.5000
6,EllipticEnv (cont=0.003),Density Estimation,0.8648,0.0131,0.0000,0.0000,0.0000
7,"Autoencoder (arch=(64, 32, 16))",Representação,0.8307,0.0168,0.0282,0.0160,0.1154
8,XGBoost + SMOTE,Classificação,0.8246,0.0152,0.0296,0.0183,0.0769
9,XGBoost + spw,Classificação,0.7526,0.0086,0.0000,0.0000,0.0000


In [21]:
# Destaques por paradigma
for paradigma in ['Density Estimation', 'Clustering', 'Representação', 'Classificação']:
    sub = df_res[df_res['paradigma'] == paradigma]
    if len(sub) > 0:
        best = sub.iloc[0]
        print(f"Melhor {paradigma}: {best['nome']} | ROC-AUC={best['roc_auc']:.4f} | PR-AUC={best['pr_auc']:.4f}")


Melhor Density Estimation: LOF (k=10) | ROC-AUC=0.9468 | PR-AUC=0.1356
Melhor Clustering: DBSCAN (eps=0.5,ms=5) | ROC-AUC=0.9379 | PR-AUC=0.0743
Melhor Representação: Autoencoder (arch=(64, 32, 16)) | ROC-AUC=0.8307 | PR-AUC=0.0168
Melhor Classificação: XGBoost + SMOTE | ROC-AUC=0.8246 | PR-AUC=0.0152


In [22]:
# Ranking completo por ROC-AUC
for i, row in df_res.iterrows():
    print(f"  {i+1}. [{row['paradigma']:<17}] {row['nome']:<38} ROC-AUC={row['roc_auc']:.4f} PR-AUC={row['pr_auc']:.4f}")


  1. [Density Estimation] LOF (k=10)                             ROC-AUC=0.9468 PR-AUC=0.1356
  2. [Clustering       ] DBSCAN (eps=0.5,ms=5)                  ROC-AUC=0.9379 PR-AUC=0.0743
  3. [Density Estimation] IF (tunado: cont=0.003,est=500)        ROC-AUC=0.9308 PR-AUC=0.0301
  4. [Density Estimation] GMM (n_comp=40)                        ROC-AUC=0.9301 PR-AUC=0.1835
  5. [Density Estimation] OC-SVM (nu=0.01,gamma=1.0)             ROC-AUC=0.9148 PR-AUC=0.0704
  6. [Clustering       ] KMeans (k=100)                         ROC-AUC=0.9098 PR-AUC=0.0556
  7. [Density Estimation] EllipticEnv (cont=0.003)               ROC-AUC=0.8648 PR-AUC=0.0131
  8. [Representação    ] Autoencoder (arch=(64, 32, 16))        ROC-AUC=0.8307 PR-AUC=0.0168
  9. [Classificação    ] XGBoost + SMOTE                        ROC-AUC=0.8246 PR-AUC=0.0152
  10. [Classificação    ] XGBoost + spw                          ROC-AUC=0.7526 PR-AUC=0.0086
  11. [Classificação    ] XGBoost (spw=610)                     

---
## 9. Análise

**Métricas honestas para anomalias raras:** com apenas 0.23% de anomalias, o F1 depende muito do threshold escolhido. Usamos **ROC-AUC e PR-AUC** (ranking, sem threshold) como métricas principais, e F1 com contaminação fixa de 1% como referência operacional.

- **Density Estimation** (LOF, IF, OC-SVM, GMM): o LOF costuma ser o melhor por modelar densidade local — detecta desvios em cada região da série, não apenas globalmente.
- **Clustering** (KMeans, DBSCAN): perde para a estimação de densidade porque modelam o **centro** dos dados, não a **borda**. Um ponto pode estar longe de clusters e ainda assim ter densidade local razoável. DBSCAN (que é clustering por densidade) tende a se sair melhor que KMeans. Clustering brilha quando há vários regimes normais bem separados.
- **Representation Learning** (Autoencoder): aprende uma representação compacta do comportamento normal e usa o **erro de reconstrução** como score. Funciona bem quando anomalias são estruturalmente diferentes dos normais (padrões que a rede não viu), mas é limitado pelo espaço latente e pela capacidade da rede. Aqui compete com os métodos clássicos sem precisar de labels.
- **Classificação Binária** (RF, XGBoost): com **split temporal**, o supervisionado fica limitado porque o treino tem pouquíssimas anomalias e a distribuição muda ao longo do tempo. Quando o split é **aleatório** (mesma distribuição entre treino/teste), o supervisionado com SMOTE **vence com folga** — mas em produção a distribuição das anomalias muda, então o não supervisionado é mais robusto.

**Conclusão:** para séries temporais com anomalias raras, a **estimação de densidade** é o paradigma mais adequado: detecta anomalias sem labels e sem assumir que o treino cobre todos os padrões de falha. O **autoencoder** (representation learning) é uma alternativa moderna interessante: não supervisionado, mas depende de ter dados normais suficientes e representativos para aprender a reconstrução.
